In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import collections
from pathlib import Path

import sys
sys.path.append('../../src/apriltag')
from read_tags import Detection as MyTag

In [ ]:
folder = Path('/Users/gerry/DropboxGatech/Painting/Graffiti_media+data/2024-02-03_iros2024_homography_data/data/')

## Load Log

In [ ]:
# Load a log
def parse_line(line):
    # Splitting the line by spaces and filtering out empty strings
    # Example line:
    #   0       1 2  3      4      5      6 7      8      9
    #   4191717 - 0: 0.0000 0.7473 1.5038 - 0.0000 0.7500 1.5000 | 0 8 1.2284 0.3533 | 0 8 0.4701 0.2876 | 0 8 -1.2426 0.8971 | 0 8 -0.2129 0.9118 | 1
    parts = [part for part in line.split() if part.strip()]
    return int(parts[0]) / 1e6, float(parts[4]), float(parts[5]), float(parts[8]), float(parts[9])

def parse_file(file):
    with open(file, 'r') as f:
        lines = f.readlines()
    t = data[:, 0]
    xy_est = data[:, 1:3]
    xy_com = data[:, 3:5]
    return t, xy_est, xy_com

line = '4191717 - 0: 0.0000 0.7473 1.5038 - 0.0000 0.7500 1.5000 | 0 8 1.2284 0.3533 | 0 8 0.4701 0.2876 | 0 8 -1.2426 0.8971 | 0 8 -0.2129 0.9118 | 1'
parsed_line = parse_line(line)
assert parsed_line == (4.191717, 0.7473, 1.5038, 0.75, 1.5), parsed_line

t, xy_est, xy_com = parse_file(folder / 'A.log')
print(t.shape, xy_est.shape, xy_com.shape)
for i in range(3):
    print(t[i] / 1e6, xy_est[i], xy_com[i])

In [ ]:
plt.plot(xy_est[:, 0], xy_est[:, 1], label='Estimated')
plt.plot(xy_com[:, 0], xy_com[:, 1], label='Commanded')

## Load AprilTag Detections

Expect:
* 1: BR
* 2: TR
* 3: TL
* 4: BL
* 9: End Effector

Tags themselves were printed 3.75" (50% scale at 7.5"), but that's black to black.  I think it might be off.

Camera was about 175" away from the tags.  
Tags are inset by paper-center away from the corner of the frame, which in turn is 4" vertically beyond the top/bottom corners (but horizontally in line).  
End effector tag is inset from BL origin corner by about x: 0.75" y: 0.625"

In [ ]:
inch2m = lambda x: x * 0.0254
m2inch = lambda x: x / 0.0254
W, H = 2.9464, 2.26
w_paper, h_paper = inch2m(8.5), inch2m(11)
w_tag, h_tag = inch2m(3.75), inch2m(3.75)
w_between_tags, h_between_tags = (W - w_paper), (H - h_paper)
print(m2inch(w_between_tags), m2inch(h_between_tags))
x_offset, y_offset = (w_paper - w_tag) / 2, (h_paper - h_tag) / 2 - inch2m(4)
print(m2inch(x_offset), m2inch(y_offset))

In [ ]:
with np.load(folder / 'camera_calibration.npz') as data:
    K = data['mtx']
    dist = data['dist']
    rvecs = data['rvecs']
    tvecs = data['tvecs']
    print(K, dist, rvecs.shape, tvecs.shape)

In [ ]:
with np.load(folder / 'A_detections_every10.npz', allow_pickle=True) as f:
    all_detections = f['all_detections'].item()
assert all(len(detections) == 5 for detections in all_detections.values()), "Not all video frames have 5 tags"

detections = all_detections[0]
print(len(all_detections), len(detections))
print()
print(detections[0])

plt.figure(figsize=(4, 3))
ts = []
for detection in detections:
    tag = MyTag(corners=detection.corners)
    R, t = tag.Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K)  # divide by 2 due to idk why
    ts.append(t)
    plt.plot(t[0], t[1], 'o')
    plt.text(t[0], t[1], f'{detection.tag_id}', fontsize=8)
    with np.printoptions(precision=3, suppress=True):
        print(R)
        print(t)
        print()


In [ ]:
with np.printoptions(precision=3, suppress=True):
    print(np.linalg.norm(ts[0] - ts[1]))
    print(np.linalg.norm(ts[1] - ts[2]))
    print(np.linalg.norm(ts[2] - ts[3]))
    print(np.linalg.norm(ts[3] - ts[0]))

In [ ]:
for detection in detections:
    print(detection.tag_id, detection.corners[0])

In [ ]:
all_trajs = {}
for trial in 'ABCD':
    with np.load(folder / f'{trial}_detections_every10.npz', allow_pickle=True) as f:
        all_detections = f['all_detections'].item()
    assert all(len(detections) == 5 for detections in all_detections.values()), "Not all video frames have 5 tags"

    def get_pos(detections):
        detections_ = {det.tag_id: det for det in detections}
        _, ee = MyTag(corners = detections_[9].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K)
        _, bl = MyTag(corners = detections_[4].corners).Rt(tag_size=3.75 / 2 * (11 / 9), camera_matrix=K)
        return np.array([ee[0] - bl[0], -(ee[1] - bl[1])])

    traj = np.array([get_pos(detections) for _, detections in sorted(all_detections.items())]).squeeze()
    ts = np.array([k / 60 for k in sorted(all_detections.keys())])

    all_trajs[trial] = (ts, traj * 0.0254)


In [ ]:
correct = np.load('data/zigzag.npy')

In [ ]:
plt.clf()
plt.plot(correct[:, 0], correct[:, 1], 'k-', label='Correct')
for trial, (ts, traj) in all_trajs.items():
    print(traj.shape)
    traj = traj * np.array([107.5 / 114.29, 77.97637795275591 / 91.39803014450392])
    print(traj.shape)
    plt.plot(traj[:, 0], traj[:, 1], label=trial)
plt.legend()